In [16]:
import pandas as pd
from collections import defaultdict
import torch
import pickle


In [12]:
mind_news = pd.read_csv(
    '/Users/harshadayiniakula/Desktop/RS/MINDsmall_train/news.tsv',
    sep='\t',
    header=None,
    names=['newid', 'vertical', 'subvertical', 'title', 'abstract', 'url', 'entities in title', 'entities in abstract'],
    dtype=str
)

In [4]:
mind_beh = pd.read_csv(
    '/Users/harshadayiniakula/Desktop/RS/MINDsmall_train/behaviors.tsv',
    sep='\t',
    header=None,
    names=['impression_id', 'user_id', 'timestamp', 'history', 'impressions'],
    dtype=str
)


In [6]:
# Drop accidental header row if present
if mind_beh['user_id'].str.lower().iloc[0] == 'userid':
    mind_beh = mind_beh.iloc[1:].reset_index(drop=True)

# Split history string into list
mind_beh['history'] = mind_beh['history'].fillna('').apply(lambda x: x.strip().split())

In [18]:
# Load the embeddings
english_embeddings = torch.load('english_article_embeddings.pt')

# Load the corresponding news IDs
with open('english_news_ids.pkl', 'rb') as f:
    english_news_ids = pickle.load(f)

/var/folders/rw/b92f531j0kg9tv8t62n84v9c0000gn/T/ipykernel_48217/2183588176.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  english_embeddings = torch.load('english_arti

In [20]:
# 1. Build mapping from news_id → row index in english_embeddings
news_id_to_index = {nid: idx for idx, nid in enumerate(mind_news['newid'])}

# 2. Create user vectors
user_vectors = {}
skipped_users = 0

for uid, clicks in zip(mind_beh['user_id'], mind_beh['history']):
    vectors = []
    for nid in clicks:
        if nid in news_id_to_index:
            idx = news_id_to_index[nid]
            vectors.append(english_embeddings[idx])
    
    if not vectors:
        skipped_users += 1
        continue

    user_vector = torch.stack(vectors).mean(dim=0)
    user_vectors[uid] = user_vector

print(f"User vectors created for {len(user_vectors)} users.")
print(f"Skipped {skipped_users} users with no valid clicked articles.")

User vectors created for 49108 users.
Skipped 3238 users with no valid clicked articles.


In [22]:
torch.save(user_vectors, 'english_user_vectors.pt')